In [1]:
import torch
from torchtext.datasets import IMDB
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import torch.nn as nn

train_iter = IMDB(split='train')

label, text = next(iter(train_iter))
print(f"Label: {label}")  # 1 = positive, 2 = negative
print(f"Text: {text[:200]}") 


Label: 1
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev


/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torchtext/datasets/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/home/aayam/micromamba/envs/RNN_env/lib/python3.11/site-packages/torchtext/vocab/__init__.py:4: 

In [2]:
tokenizer= get_tokenizer('basic_english')

def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)
        
vocab= build_vocab_from_iterator(yield_tokens(train_iter), specials=['<unk>', '<pad>'])
vocab.set_default_index(vocab['<unk>'])

print(f"Vocabulary size: {len(vocab)}")

def text_pipeline(text):
    return vocab(tokenizer(text))


def label_pipeline(label):
    return 1 if label == 'pos' else 0  

sample_text = "This movie was shit ass pathetic!"
print(f"Text: {sample_text}")
print(f"Tokens: {tokenizer(sample_text)}")
print(f"IDs: {text_pipeline(sample_text)}")

Vocabulary size: 68812
Text: This movie was shit ass pathetic!
Tokens: ['this', 'movie', 'was', 'shit', 'ass', 'pathetic', '!']
IDs: [14, 18, 17, 22531, 2000, 723, 35]


In [3]:
class TextRNN(nn.Module):
    def __init__(self,vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding= nn.Embedding(vocab_size, embed_dim)
        self.rnn= nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    
    def forward(self, text):
        embedded= self.embedding(text)
        out, hidden= self.rnn(embedded)
        out= self.fc(hidden[-1])

        return out 
    
vocab_size = len(vocab)
model = TextRNN(vocab_size=vocab_size, embed_dim=100, hidden_dim=256, output_dim=1)
print(model)


TextRNN(
  (embedding): Embedding(68812, 100)
  (rnn): RNN(100, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [4]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):

    text_list, label_list= [],[]
    max_len= 256

    for (label, text) in batch:
        label_list.append(label_pipeline(label))
        processed_text= torch.tensor(text_pipeline(text)[:max_len], dtype=torch.int64)
        text_list.append(processed_text)


    text_list= pad_sequence(text_list, batch_first=True, padding_value=vocab['<pad>'])
    label_list= torch.tensor(label_list)

    return text_list, label_list

train_list= list(IMDB(split='train'))
train_loader= DataLoader(train_list, batch_size=64, shuffle=True, collate_fn=collate_batch)

text_batch, label_batch = next(iter(train_loader))
print(f"Text batch shape: {text_batch.shape}")  
print(f"Label batch shape: {label_batch.shape}")

Text batch shape: torch.Size([64, 256])
Label batch shape: torch.Size([64])


In [5]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model= model.to(device)

criterion= nn.BCEWithLogitsLoss()

optimizer= torch.optim.AdamW(model.parameters(), lr=0.001)

num_epoch=3

for epoch in range(num_epoch):

    model.train()
    running_loss=0

    for batch_idx, (text,label) in enumerate(train_loader):

        label= label.float().unsqueeze(1).to(device)
        text= text.to(device)

        output= model(text)
        loss= criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss+= loss.item()

        if (batch_idx + 1) % 100 ==0:
            print(f"epoch {epoch+1}/{num_epoch} | batch {batch_idx+1}/{len(train_loader)} | loss {loss.item()}")

    average_loss= running_loss/len(train_loader)
    print(f"epoch {epoch+1}/{num_epoch} |  average loss {average_loss}")



epoch 1/3 | batch 100/196 | loss 0.0010024979710578918
epoch 1/3 |  average loss 0.024044325390883498
epoch 2/3 | batch 100/196 | loss 0.00017070025205612183
epoch 2/3 |  average loss 0.0007894848188197083
epoch 3/3 | batch 100/196 | loss 0.00010855495929718018
epoch 3/3 |  average loss 0.0007596914789357878


In [6]:
print(label.shape)

torch.Size([20, 1])
